# Multi-Dimensional Data

tsam_xarray handles multi-dimensional DataArrays through two mechanisms:

- **`cluster_dim`** — dimensions clustered together (shared clustering)
- **Auto-slicing** — remaining dimensions get independent clusterings

This notebook covers stacking, slicing, and weights.

In [ ]:
import plotly.io as pio
import xarray as xr
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook"

da = sample_energy_data(n_days=30)
print(f"Dims: {list(da.dims)}")
print(f"Shape: {dict(da.sizes)}")

## Multiple cluster dims

Pass `cluster_dim=["variable", "region"]` to cluster all variable-region
combinations together. They are stacked internally and unstacked in the results.

In [ ]:
da_single = da.sel(scenario="low")

result = tsam_xarray.aggregate(
    da_single,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
)
print("Result dims:", result.typical_periods.dims)
result.typical_periods.to_dataframe("value").head(10)

In [ ]:
result.typical_periods.sel(variable="solar").plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Typical periods (solar, by region)",
)

## Auto-slicing

Any dimension not in `time_dim` or `cluster_dim` is automatically sliced —
one independent aggregation per coordinate.

Here, `scenario` is auto-sliced. Each scenario gets its own clustering.

In [ ]:
result_sliced = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
)
print("Result dims:", result_sliced.typical_periods.dims)
result_sliced.cluster_weights.to_dataframe("weight")

In [ ]:
result_sliced.accuracy.rmse.to_dataframe("RMSE")

## Multiple slice dims

Only `variable` as `cluster_dim` — both `region` and `scenario` are auto-sliced.
One aggregation per (region, scenario) combination.

In [ ]:
result_multi = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)
print("Result dims:", result_multi.typical_periods.dims)
result_multi.typical_periods.sel(
    variable="solar",
    scenario="low",
).plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Typical periods (solar, low, per region)",
)

## Weights

Pass an `xr.DataArray` to weight certain coordinates higher during clustering.
Weights broadcast across missing `cluster_dim` dimensions.

In [ ]:
# Weight solar 2x, wind 1x — broadcasts across region
w = xr.DataArray(
    [2.0, 1.0, 1.0],
    dims=["variable"],
    coords={"variable": ["solar", "wind", "demand"]},
)

result_w = tsam_xarray.aggregate(
    da_single,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
    weights=w,
)
result_w.accuracy.rmse.to_dataframe("RMSE")

Weights can also span multiple dimensions:

In [ ]:
# Weight solar in north 3x
w_full = xr.DataArray(
    [[3.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]],
    dims=["region", "variable"],
    coords={
        "region": ["north", "south", "east"],
        "variable": ["solar", "wind", "demand"],
    },
)
w_full.to_dataframe("weight")